# Project 2 — Supervised Learning: Fraud Detection Pipeline

**Goal:** Build and tune classification models to identify fraudulent transactions in a highly imbalanced dataset.

### Requirements from the project brief
- Handle class imbalance using **SMOTE**
- Train **Logistic Regression** and **Random Forest**
- Use **Precision, Recall, and ROC-AUC** rather than relying on Accuracy
- Perform **hyperparameter tuning**
- Treat false positives and false negatives as important business outcomes

**Dataset:** Credit Card Fraud Detection (`creditcard.csv`)  
**Target:** `Class` — `0 = normal`, `1 = fraud`


## 1. Project workflow

`Load Data → Inspect → Clean → EDA → Train/Test Split → SMOTE on Training Data → Logistic Regression + Random Forest → Hyperparameter Tuning → Precision/Recall/ROC-AUC → Confusion Matrices → Feature Importance → Conclusion`

> **Important:** SMOTE is applied only to training data through an `imblearn` pipeline. This prevents test-set leakage.


In [ ]:
# 2. Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score, recall_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

RANDOM_STATE = 42
DATA_PATH = "../data/creditcard.csv"

print("Libraries imported successfully.")


In [ ]:
# 3. Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# 4. Basic inspection
print("Data types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isnull().sum().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
display(df["Class"].value_counts())


## 5. Data cleaning

The supplied credit-card dataset is numeric and normally has no missing values. We still check missing values and duplicates.

Duplicates are removed **before** splitting the data. This is a data-cleaning step, not SMOTE. The target column is kept unchanged.


In [ ]:
# 5. Clean data
df = df.drop_duplicates().copy()

# Remove rows with missing target/features if any appear in another copy of the dataset.
df = df.dropna().copy()

print("Shape after cleaning:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())
print("Missing values:", df.isnull().sum().sum())


In [ ]:
# 6. Exploratory Data Analysis — class imbalance
class_counts = df["Class"].value_counts().sort_index()
display(class_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values)
plt.title("Normal vs Fraud Transactions")
plt.xlabel("Class (0 = Normal, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.show()

fraud_rate = df["Class"].mean() * 100
print(f"Fraud percentage: {fraud_rate:.4f}%")


In [ ]:
# 7. Transaction amount distribution
plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="Amount", hue="Class", bins=50, element="step", stat="density", common_norm=False)
plt.title("Transaction Amount Distribution by Class")
plt.tight_layout()
plt.show()


In [ ]:
# 8. Boxplot of transaction amount
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="Class", y="Amount")
plt.title("Transaction Amount by Class")
plt.ylim(0, df["Amount"].quantile(0.99))
plt.tight_layout()
plt.show()


## 9. Prepare features and target

`X` contains predictors and `y` contains the fraud label. We use a stratified split so that the rare fraud class is represented in both training and testing sets.

**Why not apply SMOTE before the split?**  
Because synthetic information derived from the test data could leak into training and make evaluation unreliable.


In [ ]:
# 9. Features and target
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("\nTraining target distribution:")
display(y_train.value_counts())
print("\nTesting target distribution:")
display(y_test.value_counts())


## 10. Logistic Regression with SMOTE

For Logistic Regression, `RobustScaler` is useful because transaction amount and other variables can have very different scales.

The pipeline order is:

**Scaling → SMOTE → Logistic Regression**


In [ ]:
# 10. Logistic Regression pipeline
log_pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("smote", SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.5)),
    ("model", LogisticRegression(max_iter=1000, solver="liblinear", random_state=RANDOM_STATE))
])

log_pipeline.fit(X_train, y_train)

log_pred = log_pipeline.predict(X_test)
log_prob = log_pipeline.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("Precision:", precision_score(y_test, log_pred))
print("Recall:", recall_score(y_test, log_pred))
print("ROC-AUC:", roc_auc_score(y_test, log_prob))


## 11. Random Forest with SMOTE

Random Forest learns non-linear relationships using many decision trees.

To keep the notebook practical on a large dataset, the initial model uses 50 trees and SMOTE with `sampling_strategy=0.5`. You can increase the number of trees after the project works successfully on your computer.


In [ ]:
# 11. Random Forest pipeline
rf_pipeline = Pipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE, sampling_strategy=0.5)),
    ("model", RandomForestClassifier(
        n_estimators=50,
        max_depth=None,
        min_samples_split=2,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print("Random Forest")
print("Precision:", precision_score(y_test, rf_pred))
print("Recall:", recall_score(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_prob))


## 12. Hyperparameter tuning

The project brief requires tuning. We use a small grid so the notebook remains feasible on a beginner's computer.

The scoring metric is **ROC-AUC** because the assignment explicitly emphasizes ROC-AUC instead of Accuracy.


In [ ]:
# 12A. Tune Logistic Regression
log_grid = {
    "model__C": [0.1, 1, 10]
}

log_search = GridSearchCV(
    log_pipeline,
    param_grid=log_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1
)

log_search.fit(X_train, y_train)
print("Best Logistic Regression parameters:", log_search.best_params_)
print("Best CV ROC-AUC:", log_search.best_score_)


In [ ]:
# 12B. Tune Random Forest
rf_grid = {
    "model__n_estimators": [30, 50],
    "model__max_depth": [None, 15],
    "model__min_samples_split": [2, 5]
}

rf_search = GridSearchCV(
    rf_pipeline,
    param_grid=rf_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)
print("Best Random Forest parameters:", rf_search.best_params_)
print("Best CV ROC-AUC:", rf_search.best_score_)


In [ ]:
# 13. Evaluate tuned models
models = {
    "Logistic Regression": log_search.best_estimator_,
    "Random Forest": rf_search.best_estimator_
}

results = []

for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))


In [ ]:
# 14. Detailed classification reports
for name, model in models.items():
    pred = model.predict(X_test)
    print("=" * 70)
    print(name)
    print(classification_report(y_test, pred, digits=4))


In [ ]:
# 15. Confusion matrices
for name, model in models.items():
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Confusion Matrix — {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    print(name)
    print("TN, FP, FN, TP =", cm.ravel())


In [ ]:
# 16. ROC curves
plt.figure(figsize=(7, 5))

for name, model in models.items():
    prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.tight_layout()
plt.show()


## 17. Random Forest feature importance

Feature importance gives a useful explanation of which input variables contributed most to the Random Forest's decisions. In this dataset, `V1`–`V28` are anonymized PCA-derived variables, so their names do not directly describe real-world transaction properties.


In [ ]:
# 17. Feature importance
best_rf = rf_search.best_estimator_
rf_model = best_rf.named_steps["model"]

importance = pd.Series(
    rf_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False).head(15)

plt.figure(figsize=(8, 6))
importance.sort_values().plot(kind="barh")
plt.title("Top 15 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

display(importance.to_frame("Importance"))


## 18. Final interpretation

When writing the conclusion, do **not** select a model using Accuracy alone.

Discuss:
- **Precision:** How trustworthy are fraud alerts?
- **Recall:** How many real fraud cases are detected?
- **ROC-AUC:** How well does the model separate fraud from normal transactions?
- **False positives:** Legitimate transactions incorrectly flagged as fraud.
- **False negatives:** Fraudulent transactions missed by the system.

The operational choice depends on the financial institution's cost of false positives versus false negatives.


In [ ]:
# 18. Automatically generate a concise evaluation summary
display(results_df.round(4))

print("\nProject completed.")
print("Primary evaluation metrics: Precision, Recall, ROC-AUC.")
print("Accuracy is intentionally not used as the primary decision metric because the dataset is highly imbalanced.")
